# 10 — Hyperparameter tuning (D029)
- **One person per model.** Set `EXPERIMENT_ID` below to `E1`, `E2` or `E3`. Never run the same model in two sessions at once (they would write the same folders).
- **Phases** (`configs/config.yaml` → `tuning`, the same number of trials for every model):
  1. learning rate;
  2. regularisation (E1/E2: weight decay) or fine-tuning depth (E3), on top of the phase-1 winner;
  3. the phase-1/2 winner re-run with seeds 43 and 44.
- **Selection uses validation macro-F1 only. The test split is never loaded here.** The existing run of each model is the trial `base`.
- **Run all.** Finished trials are skipped and an interrupted trial resumes from its last checkpoint, so after a disconnect just Run all again.
- **Drive access for teammates:** the owner shares the folder `Deep Learning - RealWaste` with *Editor* rights; each teammate opens it in Drive → *Organize → Add shortcut* → *My Drive*, keeping the name. Otherwise set `DRIVE_ROOT` below.
- **Runtime:** GPU (T4). Images are copied from Drive to the local disk once per session (~15 min); epochs are fast afterwards.
- **Outputs (Drive):** one folder per trial in `03_Checkpoints/{id}/trials/` and `04_Results/experiments/{id}/trials/`; after phase 3 `04_Results/experiments/{id}/selection.json`, `{id}_tuning.csv`, `{id}_tuning_curves.png`.
- **When saving this notebook to GitHub**, save it as `notebooks/10_tuning_{id}.ipynb` (e.g. `10_tuning_E2.ipynb`) so teammates do not overwrite each other.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Forget src.* modules imported earlier in this session, so a re-run uses the code just pulled.
for module_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module_name]

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
EXPERIMENT_ID = "E1"   # E1, E2 or E3 - one person per model
PHASES = [1, 2, 3]     # run in order; finished trials are skipped, interrupted ones resume
DRIVE_ROOT = None      # set only if the shared folder is mounted somewhere else

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.dataset import load_frozen_split, use_local_copy
from src.tuning import BASE, plan_phase, run_trial, trial_dirs, tuning_table, winner, write_selection
from src.utils import environment_info, load_config

assert EXPERIMENT_ID in ("E1", "E2", "E3"), EXPERIMENT_ID
cfg = load_config(REPO_DIR / "configs" / "config.yaml")
if DRIVE_ROOT:
    cfg["paths"]["drive_root"] = DRIVE_ROOT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
amp = cfg["experiments"][EXPERIMENT_ID]["amp"] and device.type == "cuda"
print(json.dumps(environment_info(), indent=2))
print("device:", device, "| AMP:", amp)

cfg = use_local_copy(cfg)          # images from the local disk; checkpoints and results stay on Drive
split_df = load_frozen_split(cfg)  # SHA256-verified frozen grouped split; checks every image exists locally
print({s: int(n) for s, n in split_df["split"].value_counts().items()})
print(tuning_table(cfg, EXPERIMENT_ID).to_string(index=False))

In [ ]:
for phase in PHASES:
    for overrides, seed in plan_phase(cfg, EXPERIMENT_ID, phase, tuning_table(cfg, EXPERIMENT_ID)):
        run_trial(cfg, EXPERIMENT_ID, overrides, seed, phase, split_df, device, amp)
    print(f"\n===== {EXPERIMENT_ID}: after phase {phase}")
    print(tuning_table(cfg, EXPERIMENT_ID).to_string(index=False))

In [ ]:
table = tuning_table(cfg, EXPERIMENT_ID)
if 3 in PHASES:
    selection = write_selection(cfg, EXPERIMENT_ID)
    print(json.dumps(selection, indent=2))
    best_trial = selection["selected_trial"]
else:
    best_trial = winner(cfg, table, max(PHASES))["trial_id"]

fig, ax = plt.subplots(figsize=(11, 5))
for tid in table["trial_id"]:
    _, result_dir = trial_dirs(cfg, EXPERIMENT_ID, tid)
    history = pd.read_csv(result_dir / f"{EXPERIMENT_ID}_history.csv")
    ax.plot(history["epoch"], history["val_macro_f1"], label=tid, linewidth=2.5 if tid == best_trial else 1)
ax.set_xlabel("epoch")
ax.set_ylabel("validation macro-F1")
ax.set_title(f"{EXPERIMENT_ID} tuning - validation macro-F1 per trial (bold: selected)")
ax.grid(alpha=0.3)
ax.legend(fontsize=7)
fig.savefig(trial_dirs(cfg, EXPERIMENT_ID, BASE)[1] / f"{EXPERIMENT_ID}_tuning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(table.to_string(index=False))

## Next
1. Send the selection and the table printed above.
2. Save this notebook as `notebooks/10_tuning_{id}.ipynb` in GitHub.
3. The final test evaluation (`09_final_evaluation.ipynb`) runs only after E1, E2 and E3 all have `selection.json`.